## Data File

Downloaded the roman empire version used in the Yandex Research from https://www.kaggle.com/datasets/nbroad/wiki-20220301-en/data
Found the article from 26 parquet file by scanning them and extracting the exact document. 

In [2]:
#with open("../data/roman_empire_wikipedia_2022.txt", "r", encoding="utf-8") as f:
with open("../data/roman_empire_wikipedia_20220301.txt", "r", encoding="utf-8") as f:
    roman_empire = f.read()

## Roman - Empire Graph 

https://arxiv.org/pdf/2302.11640

This dataset is based on the Roman Empire article from English Wikipedia, which was selected since it is one of the longest articles on Wikipedia. The text was retrieved from the English Wikipedia 2022.03.01 dump from Lhoest et al. (2021). Each node in the graph corresponds to one (non-unique) word in the text. Thus, the number of nodes in the graph is equal to the article’s length. Two words are connected with an edge if at least one of the following two conditions holds:
either these words follow each other in the text, or these words are connected in the dependency tree of the sentence (one word depends on the other). Thus, the graph is a chain graph with additional shortcut edges corresponding to syntactic dependencies between words. 

In [3]:
import spacy
import networkx as nx
import torch
import numpy as np
import pandas as pd


/home/hice1/pmaji3/scratch/gnn02/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [6]:
import spacy
import networkx as nx

nlp = spacy.load("en_core_web_md")
doc = nlp(roman_empire)

# Use a Graph (undirected) to ensure A-B is only one edge
G = nx.Graph() 

#Not punctuation and space, only words
valid_tokens = [t for t in doc if not t.is_punct and not t.is_space]
token_lookup = {t.i: j for j, t in enumerate(valid_tokens)}

for j, token in enumerate(valid_tokens):
    G.add_node(j)
    
    # Sequential connection (The Chain)
    if j < len(valid_tokens) - 1:
        G.add_edge(j, j + 1)
    
    #Syntactic Dependency (The Shortcuts)
    if token.head.i in token_lookup:
        target_j = token_lookup[token.head.i]
        if target_j != j:
            # .add_edge in nx.Graph() automatically handles "at least one"
            # If the edge already exists from the chain logic, it won't add a duplicate
            G.add_edge(j, target_j)

print(f"Graph: {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")


Graph: 22600 nodes and 32684 edges.


In [7]:
print(f"Length of valid tokens is {len(valid_tokens)}")

Length of valid tokens is 22600


## Group by sentences

In [22]:
sentences = []
for sent in doc.sents:
    filtered_sent = [t for t in sent if not t.is_punct and not t.is_space]
    if filtered_sent:
        sentences.append(filtered_sent)


counter = 0
max_length = 0
min_length = 100
for sent in sentences:
    if len(sent) < min_length:
        min_length = len(sent)
    if len(sent) > max_length:
        max_length = len(sent)
    counter += len(sent)
print("Verifying that the # of tokens in the sentences",counter)
print(f"The max length of the sentence is {max_length}")
print(f"The min length of the sentence is {min_length}")

print(f"Length of sentences is {len(sentences)}")
for sent in sentences:
    print(sent)

Verifying that the # of tokens in the sentences 22600
The max length of the sentence is 131
The min length of the sentence is 2
Length of sentences is 923
[The, Roman, Empire, was, the, post, Republican, period, of, ancient, Rome]
[As, a, polity, it, included, large, territorial, holdings, around, the, Mediterranean, Sea, in, Europe, Northern, Africa, and, Western, Asia, ruled, by, emperors]
[From, the, accession, of, Caesar, Augustus, as, the, first, Roman, emperor, to, the, military, anarchy, of, the, 3rd, century, it, was, a, principate, with, the, city, of, Rome, as, sole, capital]
[Later, the, empire, was, ruled, by, multiple, emperors, who, shared, rule, over, the, Western, Roman, Empire, and, over, the, Eastern, Roman, Empire]
[Rome, remained, the, nominal, capital, of, both, parts, until, AD, 476, when, the, imperial, insignia, were, sent, to, Constantinople, following, the, capture, of, the, Western, capital, Ravenna, by, the, barbarians, of, Odoacer, and, the, subsequent, dep

In [24]:
all_words = []
for sent in sentences:
    for t in sent:
        all_words.append(t.text.lower())

vocab = {"[PAD]": 0, "[MASK]": 1, "[UNK]" : 2}



for word in sorted(set(all_words)):
    if word not in vocab:
        vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")
print(vocab)

Vocab size: 4895
{'[PAD]': 0, '[MASK]': 1, '[UNK]': 2, "'s": 3, '1': 4, '1.2–1.7': 5, '1.5': 6, '10': 7, '10,000': 8, '100': 9, '100,000': 10, '12': 11, '120': 12, '125,000': 13, '12th': 14, '132': 15, '135': 16, '13th': 17, '14': 18, '1453': 19, '15': 20, '15,000': 21, '150': 22, '150,000': 23, '15–16': 24, '160': 25, '166': 26, '17': 27, '17.2': 28, '17.9': 29, '1712)—hero': 30, '177': 31, '177–192': 32, '17th': 33, '180': 34, '1861': 35, '18th': 36, '193–235': 37, '19th': 38, '1st': 39, '1st–3rd': 40, '2': 41, '2,000': 42, '20': 43, '200': 44, '200,000': 45, '200,000–250,000': 46, '212': 47, '222–235': 48, '24': 49, '246–251': 50, '25': 51, '250': 52, '26': 53, '268': 54, '27': 55, '270–275': 56, '271': 57, '28': 58, '284': 59, '284–305': 60, '286': 61, '29': 62, '2nd': 63, '3,000': 64, '3,500': 65, '30': 66, '300': 67, '300,000': 68, '303': 69, '31': 70, '311': 71, '313': 72, '33': 73, '331–420': 74, '336': 75, '35': 76, '374': 77, '380': 78, '394': 79, '395': 80, '397': 81, '3rd':

# Convert sentences to IDs

In [31]:
input_ids = []
for sent in sentences:
        ids = [vocab.get(t.text.lower(), vocab["[UNK]"]) for t in sent]
        input_ids.append(ids)    
print(input_ids)

[[4448, 3859, 1542, 4792, 4448, 3410, 3757, 3280, 3071, 287, 3870], [391, 129, 3375, 2441, 2315, 2534, 4434, 2192, 377, 4448, 2799, 3973, 2310, 1644, 3016, 228, 288, 4813, 392, 3889, 655, 1539], [1945, 4448, 152, 3071, 660, 456, 391, 4448, 1849, 3859, 1538, 4503, 4448, 2851, 283, 3071, 4448, 82, 749, 2441, 4792, 129, 3478, 4842, 4448, 819, 3071, 3870, 391, 4153, 687], [2543, 4448, 1542, 4792, 3889, 655, 2937, 1539, 4822, 4055, 3888, 3163, 4448, 4813, 3859, 1542, 288, 3163, 4448, 1487, 3859, 1542], [3870, 3729, 4448, 3007, 687, 3071, 604, 3219, 4663, 186, 95, 4817, 4448, 2292, 2379, 4811, 4012, 4503, 1028, 1874, 4448, 692, 3071, 4448, 4813, 687, 3626, 655, 4448, 503, 3071, 3070, 288, 4448, 4308, 1280, 3071, 3871, 455], [4448, 201, 3071, 794, 391, 4448, 4247, 801, 3071, 4448, 3859, 1542, 2310, 186, 78, 288, 4448, 1763, 3071, 4448, 4813, 3859, 1542, 4503, 2030, 2499, 1075, 2754, 4448, 1558, 3071, 833, 304, 288, 4448, 532, 3071, 4448, 2840, 237], [527, 3071, 4473, 1651, 257, 4842, 4448, 20

## Padding to equal length

In [32]:
import torch
from torch.nn.utils.rnn import pad_sequence

input_tensors = [torch.tensor(ids) for ids in input_ids]
padded_inputs = pad_sequence(input_tensors, batch_first=True, padding_value=0)

print(f"padded shape: {padded_inputs.shape}")

padded shape: torch.Size([923, 131])


# Encode the Node Feature

## Identify the Node Class

In [26]:
from collections import Counter

# Get all raw syntactic roles from your valid_tokens list
raw_roles = [token.dep_ for token in valid_tokens]

role_counts = Counter(raw_roles)

# Identify the 17 most frequent roles
top_17_roles = [role for role, count in role_counts.most_common(17)]

for i, role in enumerate(top_17_roles):
    print(f"Class {i}: {role} ({role_counts[role]} occurrences)")

Class 0: pobj (3162 occurrences)
Class 1: prep (3137 occurrences)
Class 2: det (2510 occurrences)
Class 3: amod (2376 occurrences)
Class 4: conj (1383 occurrences)
Class 5: nsubj (1224 occurrences)
Class 6: cc (1080 occurrences)
Class 7: compound (928 occurrences)
Class 8: ROOT (923 occurrences)
Class 9: dobj (872 occurrences)
Class 10: advmod (795 occurrences)
Class 11: aux (446 occurrences)
Class 12: auxpass (370 occurrences)
Class 13: appos (362 occurrences)
Class 14: nsubjpass (329 occurrences)
Class 15: poss (321 occurrences)
Class 16: relcl (296 occurrences)


In [27]:
number_roles = [t.dep_ for t in valid_tokens if t.like_num]
num_role_counts = Counter(number_roles)

print("Roles assigned to numbers:")
for role, count in num_role_counts.items():
    print(f"{role}: {count}")

Roles assigned to numbers:
amod: 91
nummod: 183
pobj: 39
compound: 20
attr: 13
quantmod: 11
conj: 15
nsubjpass: 1
ccomp: 1
dobj: 4
appos: 8
poss: 3
advmod: 3
nsubj: 2
oprd: 1
npadvmod: 1
prep: 1


In [12]:
role_to_class = {role: i for i, role in enumerate(top_17_roles)}

node_labels = []
for role in raw_roles:
    # Get class ID from mapping, default to 17 (18th class) if not in top 17
    class_id = role_to_class.get(role, 17)
    node_labels.append(class_id)

import numpy as np
y = np.array(node_labels)

print(f"\nNode labels shape: {y.shape}")
print(f"Unique classes found: {np.unique(y)}")


Node labels shape: (22600,)
Unique classes found: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]


# Generate the Edge Index

In [13]:
edges = list(G.edges())
edge_list_df = pd.DataFrame(edges, columns=['source', 'target'])
edge_list_df = edge_list_df.astype(int)
edge_list_df.to_csv('../data/edge_list.txt', sep=' ', index=False, header=False)


# Create Labels

In [14]:
import pandas as pd
import numpy as np

labels_df = pd.Series(y)

# Save to labels.csv
labels_df.to_csv('../data/labels.csv', index=False, header=False)


# Generate NPZ

In [26]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from scripts.create_npz import create_npx

create_npx.create_npz_dataset(node_features_file="../data/node_features_roberta.csv", labels_file='../data/labels.csv', edges_file='../data/edge_list.txt', num_splits=10, output_file_name='../data/roman-roberta.npz')


Verified Split 0 - Train: 13560, Val: 4520, Test: 4520
no of nodes 22600
no of features per node 768
no of node labels, should matach no of nodes 22600
no of edges 32684
